# Ask the Docs — RAG Assistant

**Goal:** Answer questions using a small collection of public Python documentation pages, cite the source chunks, and abstain when the answer is not supported by the documents.

## Pipeline
1. Load five public Python documentation pages.
2. Split page text into overlapping word chunks.
3. Embed chunks with `all-MiniLM-L6-v2`.
4. Retrieve similar chunks with FAISS.
5. Ask Gemini to answer only from retrieved evidence and cite source chunk IDs.
6. Evaluate retrieval separately from answer quality.

> **Before running:** In Google Colab, add your Gemini API key under **Secrets** with the name `GEMINI_API_KEY` and enable notebook access. Run cells from top to bottom. Gemini free-tier quotas can limit answer generation; the retrieval pipeline does not need the API key.

## 1. Install dependencies
Run this cell once in a fresh Colab runtime.

In [ ]:
# Install the libraries used for scraping, embeddings, retrieval, and Gemini.
!pip -q install sentence-transformers faiss-cpu requests beautifulsoup4 google-genai pandas

## 2. Load the source documents
We use five public pages from the official Python tutorial. Each document keeps its title and URL for citation.

In [ ]:
import requests
from bs4 import BeautifulSoup

# Public documentation pages used as the knowledge base.
documents = {
    "Python Tutorial": "https://docs.python.org/3/tutorial/index.html",
    "Python Data Structures": "https://docs.python.org/3/tutorial/datastructures.html",
    "Python Functions": "https://docs.python.org/3/tutorial/controlflow.html",
    "Python Errors and Exceptions": "https://docs.python.org/3/tutorial/errors.html",
    "Python Classes": "https://docs.python.org/3/tutorial/classes.html",
}

all_documents = []

for title, url in documents.items():
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "AskTheDocsRAG/1.0"}
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove page elements that are not part of the main documentation text.
    for element in soup.select(
        "nav, header, footer, script, style, .sphinxsidebar, .related"
    ):
        element.decompose()

    main = soup.select_one("main") or soup.body
    text = main.get_text(separator="\n", strip=True)

    all_documents.append({
        "title": title,
        "url": url,
        "text": text
    })

print(f"Loaded {len(all_documents)} documents.")
for doc in all_documents:
    print(f"- {doc['title']}: {len(doc['text'])} characters")

## 3. Chunk the documents
Each chunk contains up to 180 words, with 40 words overlapping the next chunk. Overlap helps preserve context across chunk boundaries.

In [ ]:
def chunk_text(text, chunk_size=180, overlap=40):
    """Split text into word-based chunks with a configurable overlap."""
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break

    return chunks


# Store each chunk with a stable ID and its original source metadata.
all_chunks = []

for doc in all_documents:
    chunks = chunk_text(doc["text"], chunk_size=180, overlap=40)
    safe_title = doc["title"].replace(" ", "_")

    for chunk_number, chunk in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{safe_title}_{chunk_number}",
            "title": doc["title"],
            "url": doc["url"],
            "text": chunk
        })

print(f"Created {len(all_chunks)} chunks.")
print("First chunk preview:", all_chunks[0]["text"][:350])
print("First chunk source:", all_chunks[0]["url"])

## 4. Embed the chunks
`all-MiniLM-L6-v2` creates a 384-dimensional vector for each chunk. Normalized embeddings make inner-product search equivalent to cosine similarity.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the sentence embedding model.
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("Embedding count:", embeddings.shape[0])
print("Embedding dimensions:", embeddings.shape[1])
print("All embeddings finite:", bool(np.isfinite(embeddings).all()))

## 5. Build the FAISS index and retrieval function
The index returns the top-k chunks most semantically similar to a question. Similarity scores are used as a simple evidence threshold, not as a calibrated probability.

In [ ]:
import faiss

# IndexFlatIP uses inner product; normalized vectors make it cosine-similarity search.
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def retrieve_chunks(question, top_k=5):
    """Return the top-k document chunks ranked by cosine similarity."""
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        question_embedding,
        min(top_k, len(all_chunks))
    )

    results = []
    for score, chunk_idx in zip(scores[0], indices[0]):
        if chunk_idx < 0:
            continue

        chunk = all_chunks[int(chunk_idx)]
        results.append({
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "url": chunk["url"],
            "text": chunk["text"],
            "score": float(score)
        })

    return results

print(f"FAISS index contains {index.ntotal} chunks.")

## 6. Check retrieval before using the LLM
This sanity check confirms that relevant source documents appear among the top three results. It is a retrieval check, **not** a measure of final answer accuracy.

In [ ]:
# Try one in-scope question and inspect the retrieved evidence.
sample_question = "How does Python handle exceptions?"

for rank, item in enumerate(retrieve_chunks(sample_question, top_k=3), start=1):
    print(f"\nResult {rank}: {item['title']}")
    print("Chunk ID:", item["chunk_id"])
    print("Similarity:", round(item["score"], 3))
    print("Source:", item["url"])
    print(item["text"][:300].replace("\n", " "))

## 7. Configure Gemini
Create a Colab Secret named `GEMINI_API_KEY` and allow this notebook to access it. This cell only configures the client; it does not send a request or spend generation quota.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai.errors import APIError
import time
import re

# Keep the API key in Colab Secrets; do not paste it into notebook code.
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it in Colab Secrets and enable notebook access."
    )

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.8-flash"

print("Gemini client configured. No API request has been made yet.")

## 8. Add retry handling for temporary server errors
Temporary 5xx errors are retried with short exponential backoff. A 429 quota/rate-limit error is raised immediately so the notebook does not waste time retrying a daily quota exhaustion.

In [ ]:
def generate_with_retry(prompt, attempts=3):
    """Call Gemini; retry transient server errors, but do not retry quota errors."""
    for attempt in range(attempts):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={"temperature": 0}
            )
        except APIError as error:
            code = getattr(error, "code", None)

            # 429 often means the quota/rate limit is exhausted; retrying immediately
            # will not help when the daily free-tier quota has been reached.
            if code == 429:
                raise

            if code not in (500, 502, 503, 504) or attempt == attempts - 1:
                raise

            wait_seconds = 2 ** (attempt + 1)
            print(f"Temporary Gemini server error ({code}); retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

## 9. Answer questions with evidence and citations
The function retrieves chunks, filters them by a configurable similarity threshold, and asks Gemini to answer only from those excerpts. The returned citation metadata is restricted to source IDs that the model actually cites in its answer.

In [ ]:
ABSTENTION_MESSAGE = "I could not find sufficient information in the documents."

def answer_question(question, top_k=5, min_similarity=0.35):
    """Answer from retrieved excerpts or abstain when evidence is insufficient."""
    retrieved = retrieve_chunks(question, top_k=top_k)
    evidence = [item for item in retrieved if item["score"] >= min_similarity]

    if not evidence:
        return {"answer": ABSTENTION_MESSAGE, "citations": []}

    context = "\n\n".join(
        f"[SOURCE ID: {item['chunk_id']}]\n"
        f"[TITLE: {item['title']}]\n"
        f"[URL: {item['url']}]\n"
        f"[TEXT]\n{item['text']}"
        for item in evidence
    )

    prompt = f"""
You are an assistant that answers questions using ONLY the document excerpts below.
Rules:
1. Do not use outside knowledge.
2. If the excerpts do not contain enough evidence, reply exactly: NOT_FOUND
3. Do not invent facts or source IDs.
4. Give a concise answer supported by the excerpts.
5. Cite every source you use inline using exactly this format:
   [SOURCE ID: exact_chunk_id]
6. Use only source IDs included in the excerpts.

DOCUMENT EXCERPTS:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = generate_with_retry(prompt)
    answer = (response.text or "").strip()

    if not answer or answer == "NOT_FOUND" or answer.startswith("NOT_FOUND"):
        return {"answer": ABSTENTION_MESSAGE, "citations": []}

    # Extract only valid source IDs actually cited in the generated answer.
    cited_ids = re.findall(r"\[SOURCE ID:\s*([^\]]+)\]", answer)
    evidence_by_id = {item["chunk_id"]: item for item in evidence}
    # Preserve the order in which sources appear in the answer and remove duplicates.
    valid_cited_ids = list(dict.fromkeys(
        source_id for source_id in cited_ids if source_id in evidence_by_id
    ))

    citations = [
        {
            "chunk_id": source_id,
            "title": evidence_by_id[source_id]["title"],
            "url": evidence_by_id[source_id]["url"],
            "score": round(evidence_by_id[source_id]["score"], 3)
        }
        for source_id in valid_cited_ids
    ]

    # Flag a response with no valid citation rather than pretending a source was cited.
    if not citations:
        answer += "\n\n[Warning: no valid source citation was detected in the generated answer.]"

    return {"answer": answer, "citations": citations}

print("answer_question() is ready.")

## 10. Try one end-to-end question
**Note:** This cell sends a Gemini API request. If the free-tier daily quota is exhausted, wait for quota reset or configure another available model/provider.

In [ ]:
# One end-to-end test: retrieve evidence, generate an answer, and display cited sources.
# Catch API/quota errors so later retrieval-only evaluation cells can still run.
try:
    result = answer_question("How does Python handle exceptions?")

    print("ANSWER:\\n")
    print(result["answer"])
    print("\\nCITATIONS:")
    for citation in result["citations"]:
        print(f"- {citation['title']} | {citation['chunk_id']}")
        print(citation["url"])
except Exception as error:
    print("The end-to-end generation test could not complete.")
    print("This may be caused by Gemini API quota or temporary availability.")
    print("Error:", str(error)[:500])
    print("You can still run the retrieval evaluation below; it does not call Gemini.")

## 11. Evaluate retrieval on 10 questions
This test does not call Gemini. It checks whether the expected document appears in the top three retrieved chunks for nine in-scope questions, and whether the out-of-scope France question is below the configured similarity threshold. The result is **retrieval-check pass rate**, not answer accuracy.

In [ ]:
import pandas as pd

retrieval_test_cases = [
    ("How does Python handle exceptions?", "Python Errors and Exceptions"),
    ("What is the purpose of the else clause?", "Python Errors and Exceptions"),
    ("How can multiple exceptions be handled?", "Python Errors and Exceptions"),
    ("What is a Python list?", "Python Data Structures"),
    ("How do you add an item to a list?", "Python Data Structures"),
    ("What is a Python dictionary?", "Python Data Structures"),
    ("What is a function in Python?", "Python Functions"),
    ("What is a lambda function?", "Python Functions"),
    ("How do you define a class in Python?", "Python Classes"),
    ("What is the capital of France?", None),
]

retrieval_rows = []

for question, expected_source in retrieval_test_cases:
    retrieved = retrieve_chunks(question, top_k=3)
    top_score = retrieved[0]["score"] if retrieved else 0.0
    top_source = retrieved[0]["title"] if retrieved else None

    if expected_source is None:
        passed = top_score < 0.35
    else:
        passed = any(
            item["title"] == expected_source and item["score"] >= 0.35
            for item in retrieved
        )

    retrieval_rows.append({
        "Question": question,
        "Expected source": expected_source or "No answer in docs",
        "Top source": top_source,
        "Top score": round(top_score, 3),
        "Retrieval check passed": passed
    })

retrieval_df = pd.DataFrame(retrieval_rows)
display(retrieval_df)

passed_count = int(retrieval_df["Retrieval check passed"].sum())
total_count = len(retrieval_df)
print(f"Retrieval checks passed: {passed_count}/{total_count}")
print(f"Retrieval check pass rate: {passed_count / total_count:.0%}")

# Save the detailed retrieval results for the submission.
retrieval_df.to_csv("retrieval_evaluation.csv", index=False)
print("Saved retrieval_evaluation.csv")

## 12. Answer-quality evaluation (run once quota is available)

This cell sends up to 10 Gemini requests. Run it **only when the API quota is available**. The expected answer is a reference criterion, not an exact-match string. Review each actual answer and set `Correct after review` to `True` or `False`; API errors remain unscored.

The code intentionally does not claim an answer-accuracy score until the answers have been reviewed.

In [ ]:
answer_test_cases = [
    {
        "question": "How does Python handle exceptions?",
        "expected_answer": "Explain try/except handling and what happens when an exception occurs."
    },
    {
        "question": "What is the purpose of the else clause in a try statement?",
        "expected_answer": "The else clause runs when the try clause completes without an exception."
    },
    {
        "question": "How can multiple exceptions be handled?",
        "expected_answer": "Use multiple except clauses for different exception types."
    },
    {
        "question": "What is a Python list?",
        "expected_answer": "A list is an ordered, mutable sequence."
    },
    {
        "question": "How do you add an item to the end of a list?",
        "expected_answer": "Use the append() method."
    },
    {
        "question": "What is a Python dictionary?",
        "expected_answer": "A dictionary stores key-value pairs."
    },
    {
        "question": "What is a function in Python?",
        "expected_answer": "A function is a named, reusable block of code, commonly defined with def."
    },
    {
        "question": "What is a lambda function?",
        "expected_answer": "A lambda expression creates a small anonymous function."
    },
    {
        "question": "How do you define a class in Python?",
        "expected_answer": "Use the class keyword."
    },
    {
        "question": "What is the capital of France?",
        "expected_answer": "Abstain because this is not answered by the Python documents."
    },
]

answer_rows = []

for number, test in enumerate(answer_test_cases, start=1):
    print(f"Question {number}/{len(answer_test_cases)}: {test['question']}")
    row = {
        "Question": test["question"],
        "Expected answer / criterion": test["expected_answer"],
        "Actual answer": "",
        "Citations": "",
        "Status": "",
        "Correct after review": None
    }

    try:
        result = answer_question(test["question"])
        row["Actual answer"] = result["answer"]
        row["Citations"] = "; ".join(c["chunk_id"] for c in result["citations"])
        row["Status"] = "completed"
    except Exception as error:
        row["Actual answer"] = str(error)[:300]
        row["Status"] = "API_ERROR - not scored"

    answer_rows.append(row)

answer_evaluation_df = pd.DataFrame(answer_rows)
display(answer_evaluation_df)

# Save raw outputs. Review answers before calculating an answer-quality score.
answer_evaluation_df.to_csv("answer_evaluation.csv", index=False)
print("Saved answer_evaluation.csv")
print("Review the completed answers and fill 'Correct after review' before reporting accuracy.")

## 13. Evaluation notes and observed development issues

### Retrieval result observed during development
The 10-question retrieval check passed **10/10 (100%)** with the checks defined above. This is a small, hand-written test set and should not be interpreted as a general retrieval benchmark.

### Two real issues observed during development
1. **False abstention on “What is a Python list?”** Relevant `Python Data Structures` chunks were retrieved (top similarity was about `0.586`), but an earlier answer-generation run returned the abstention message. This is a generation/grounding failure despite relevant retrieval.
2. **Over-inclusive citation metadata in the earlier implementation.** The first version returned every retrieved evidence chunk in the citation list, including chunks the generated answer did not cite. The cleaned `answer_question()` now filters citation metadata to source IDs explicitly cited in the answer. Citation correctness still needs manual review.

### Limitations and possible improvements
- The similarity threshold (`0.35`) is a starting heuristic, not a calibrated confidence score.
- The corpus contains only five Python tutorial pages; answers outside this scope should be declined.
- The current evaluation set is small. Add more in-scope and out-of-scope questions and review answer correctness and citation support manually.
- Improve text cleaning, chunk boundaries, and retrieval using a reranker or hybrid keyword + vector search.
- API quota errors are infrastructure/evaluation interruptions, not incorrect model answers. Do not count them as wrong answers; report them separately.